# Sesión 3 · Ejercicio 2 — Control fino
**Objetivo:** medir, con un modelo de difusión ya preentrenado, las dos
palancas del muestreo: **pasos** (calidad ↔ tiempo) y **guía**
(adherencia a la condición ↔ diversidad).
**Tiempo:** MÍNIMO 30 min · COMPLETO 60 min
**Produce:** §6 de su bitácora — la rejilla del barrido + su lectura
**Necesitas:** la hipótesis que dejó en el chat antes del receso y
copió a su §6


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
# Si ya había una copia en esta máquina, se pone al día: de lo
# contrario se quedaría con la versión del día que la clonó, y las
# correcciones publicadas después nunca le llegarían.
!git pull -q --ff-only
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


### Paso 1 · Hoy usted es piloto de pruebas, no mecánico

En el Ejercicio 1 construyó un difusor desde cero. Éste ya viene
**entrenado por la instructora** sobre el conjunto de demostración de
imágenes: usted no va a entrenar nada — va a **medir** cómo responde
el modelo a sus dos palancas de muestreo.

Al cargar el checkpoint puede aparecer una figura «gratis»
(viene guardada dentro del archivo): ignórela, no es un error.


In [ ]:
#### OBLIGATORIO #### — cargar el modelo de difusión preentrenado
# Es la misma clase Difusion del módulo, preentrenada por la
# instructora sobre el conjunto de demostración (condicional a la
# clase, con guía sin clasificador). En Colab tarda ~1 min en bajar.
X, y, meta = datos.cargar("imagen")
contenido, _ = rescate.cargar("s3_difusion_preentrenada")
difusion = modelos.Difusion(meta, pasos=contenido["pasos"])
difusion.red.load_state_dict(contenido["state_dicts"]["red"])
print("Modelo preentrenado listo (no se entrena nada en este ejercicio).")


### Paso 2 · La referencia — y el precio de la familia

Antes de mover palancas, una tanda con los valores por defecto:
**200 pasos, guía 3**. Va a generar 16 imágenes de la clase
minoritaria (`anillo`) y va a tardar ~1 minuto — es la celda más
lenta del cuaderno, **a propósito**: 200 pasos significa pasar por la
red 200 veces por imagen. Ése es el precio de la difusión, sentido en
carne propia.

No espere fotografías: son anillos de 32×32 de un modelo pequeño —
granulosos y con algún defecto. La comparación que importa es **entre
configuraciones**, no contra un generador comercial.


In [ ]:
#### OBLIGATORIO #### — una salida con los parámetros por defecto
# guia=3 y todos los pasos del entrenamiento (200): la referencia.
CLASE = meta["nombres_clases"].index(meta["clase_minoritaria"])
graficas.rejilla(difusion.muestrear(16, y=CLASE, guia=3.0),
                 f"Referencia: clase {meta['clase_minoritaria']!r}, "
                 "200 pasos, guía 3")


### Paso 3 · Palanca 1 — ¿cuántos pasos son suficientes?

DDIM permite recorrer el camino de regreso con **menos pasos, más
grandes**, sin reentrenar. La pregunta es cuánto se pierde. Va a
generar 8 muestras con 5, 10, 25 y 50 pasos, **con cronómetro**.

Qué mirar: a este tamaño de imagen la diferencia visual entre 5 y 50
pasos es **sutil** (todas producen anillos; con más pasos mejora la
textura fina). La diferencia dramática de esta palanca está en el
**reloj**, no en el ojo — por eso se mide el tiempo.


In [ ]:
#### OBLIGATORIO #### — palanca 1: ¿cuántos pasos de muestreo?
import time
# COMPLETAR: para cada valor de PASOS, genere 8 muestras de su clase
# con ese número de pasos y guarde el tiempo que tardó.
# Pista: difusion.muestrear(8, y=CLASE, pasos=p, guia=3.0)
#        y time.time() antes y después.
PASOS = [5, 10, 25, 50]
muestras_p, tiempos = {}, {}
for p in PASOS:
    inicio = time.time()
    muestras_p[p] = ...
    tiempos[p] = time.time() - inicio
    graficas.rejilla(muestras_p[p], f"{p} pasos", n=8)


In [ ]:
#### OBLIGATORIO #### — el precio en tiempo de cada configuración
for p in PASOS:
    print(f"  {p:3d} pasos → {tiempos[p]:6.2f} s por lote de 8 "
          f"({tiempos[p] / 8:.3f} s por muestra)")
print("La relación es lineal: cada paso es una pasada por la red.")


### Paso 4 · Palanca 2 — ¿cuánta obediencia?

La guía regula cuánto obedece el modelo a la condición «genera un
anillo». Va a barrer guía 1, 3, 7.5 y 15, siempre con 50 pasos.

Qué mirar: con guía baja hay **variedad** — y algún «colado» que no
es anillo; con guía alta, **todos** son anillos… pero cada vez más
parecidos entre sí, y en el extremo aparecen artefactos. Obediencia a
cambio de variedad. (¿De qué fenómeno de la Sesión 2 es pariente
esto? Guarde la respuesta para el contraste final.)


In [ ]:
#### OBLIGATORIO #### — palanca 2: ¿cuánta guía?
# COMPLETAR: genere 8 muestras de su clase con 50 pasos para cada
# valor de guía. ¿Qué pasa en los extremos?
# Pista: el parámetro se llama guia (guidance scale). Con guia=1 el
#        modelo es condicional puro; valores altos fuerzan la clase.
GUIAS = [1.0, 3.0, 7.5, 15.0]
muestras_g = {}
for g in GUIAS:
    muestras_g[g] = ...
    graficas.rejilla(muestras_g[g], f"guía {g}", n=8)


### Paso 5 · Las dos palancas a la vez — su figura de bitácora

El experimento más controlado del módulo: **una** muestra por cada
combinación (pasos, guía), todas partiendo del **mismo ruido
inicial** (semilla fija). Así, lo único que cambia entre casillas son
sus dos palancas — cualquier diferencia es atribuible a ellas.

La rejilla resultante (4×4) es la figura de su §6. Léala por ejes:
horizontal = pasos (afina detalle, se paga en tiempo); vertical =
guía (impone la clase, se paga en variedad).


In [ ]:
#### OBLIGATORIO #### — las dos palancas cruzadas, en una sola figura
# Una muestra por combinación (pasos, guía), con el MISMO ruido inicial
# en cada renglón para que la comparación sea justa.
import torch
cruzado = {}
for p in PASOS:
    for g in GUIAS:
        torch.manual_seed(7)
        cruzado[(p, g)] = difusion.muestrear(1, y=CLASE, pasos=p,
                                             guia=g)[0]
fig = graficas.rejilla_barrido(cruzado, eje_x="pasos", eje_y="guía")
fig.savefig("bitacora_s3e2_barrido.png", dpi=120)


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿No corrió el barrido? Esto trae la rejilla precomputada y los
# tiempos medidos, y puede pasar directo a la lectura.
contenido, figuras = rescate.cargar("s3_barrido")
tiempos = contenido["tiempos"]
PASOS = sorted(tiempos)
for p in PASOS:
    print(f"  {p:3d} pasos → {tiempos[p]:6.2f} s por lote de 8")
# La figura precomputada queda con el nombre que pide la bitácora.
figuras[0].savefig("bitacora_s3e2_barrido.png", dpi=120)
print("bitacora_s3e2_barrido.png guardada.")


### Contraste con su hipótesis — dos renglones

Antes del receso usted dejó en el chat (y copió a su §6) una
predicción de qué pasaría al subir la guía y al bajar los pasos.
Hora de confrontarla con sus propios datos.

- **Lo que predije:** ___
- **Lo que observé:** ___
- ¿A partir de cuántos pasos ya no nota mejora visual? ___
- Con guía 15: ¿las 8 muestras son más parecidas entre sí que con
  guía 1? ___ (¿de qué fenómeno de la Sesión 2 es pariente eso?)


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §6 de su bitácora y adjunte")
print("bitacora_s3e2_barrido.png:\n")
print(f"- Clase generada: {meta['clase_minoritaria']!r}")
print(f"- Tiempo por muestra a 50 pasos: {tiempos[50] / 8:.3f} s")
print("- Pasos suficientes a la vista: <...>")
print("- Efecto de la guía alta: <...>")


### EXTENSIÓN (equipos rápidos)
Repita el barrido cruzado con OTRA clase del conjunto (una mayoritaria).
¿Las palancas se comportan igual cuando el modelo vio muchos más
ejemplos de la clase? ¿La guía alta «inventa» menos?
